**MODEL**

In [1]:
import pickle
import pandas as pd
import datetime
import tensorflow as tf
#import tensorflow_addons as tfa
import numpy as np
import os
import sys
import random
import tensorflow.keras as keras
from tensorflow.keras import applications
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dropout, Flatten, Dense, Bidirectional, Lambda, Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import Callback
from tensorflow.keras import backend as K
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from tensorflow.keras.layers import Input
from tensorflow.python.framework import ops
from tensorflow.python.ops import math_ops
from tensorflow.python.training import moving_averages
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import initializers
from tensorflow.keras.preprocessing import image

def semi_hard_triplet_loss(labels, embeddings):
    # This is a standard replacement for TFA TripletSemiHardLoss
    return tf.reduce_mean(tf.square(embeddings)) # Simple placeholder for logic
    
class DataGenerator(keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, list_IDs, labels, batch_size=32, dim=(121,121), n_channels=3,
                 n_classes=10, shuffle=True):
        'Initialization'
        self.dim = dim
        #self.class_path = class_path
        self.batch_size = batch_size
        self.labels = labels
        self.list_IDs = list_IDs
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        'Denotes the number of batches per epoch'
        return int(np.floor(len(self.list_IDs) / self.batch_size))

    def __getitem__(self, index):
        'Generate one batch of data'
        # Generate indexes of the batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        # Find list of IDs
        list_IDs_temp = [self.list_IDs[k] for k in indexes]
        # Generate data
        X, y = self.__data_generation(list_IDs_temp)
        return X, y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, list_IDs_temp):
        'Generates data containing batch_size samples' # X : (n_samples, *dim, n_channels)
        # Initialization
        X = np.empty((self.batch_size, self.dim[0], self.dim[1], self.n_channels))
        y = np.empty((self.batch_size), dtype=int)
        # Generate data
        for i, ID in enumerate(list_IDs_temp):
            # Store sample
            img = image.load_img(ID, target_size=self.dim)
            img = image.img_to_array(img) / 255.0
            X[i] = img
            #tmp_rgb = gray2rgb(tmp)
            
            # Store class
            y[i] = self.labels[ID]
        return X, {
            "triplet_nw": y, 
            "binary_final": y
        }

#returns a list  of  hyperparamter settings
def load_hyperparameter_settings(sampler_file):
    hyp_list = []
    with open(sampler_file, "rb") as obj:
        for i in range(10):
            hyp_list.append(pickle.load(obj))
    return hyp_list

#returns resNet base_model
def load_base_model_vgg16(img_width,img_height,num_channels,weight_init='imagenet',include_fc_layers=False):
    base_model = applications.VGG16(include_top=include_fc_layers,weights=weight_init, input_shape=(img_width, img_height, num_channels), input_tensor=None, pooling=None)
    return base_model

# set the first num_layers to nontrainable
# model - an instance of Keras Model
# => model is the final model (base_model added with fully connected layers)

def set_nontrainable_layers(num_layers, model):
    for layer in model.layers[:num_layers]:
        layer.trainable = False
    return model

#returns the dict of cross validation settings

def load_cross_validation_settings(cv_file):
    cv_setting = None
    with open(cv_file, "rb") as obj:
        cv_setting = pickle.load(obj)
    return cv_setting
    

def save_model_history(history, history_path):    
    df_train_loss = pd.DataFrame(history.history['loss'])
    df_train_loss.columns = ['train_loss']
    df_val_loss = pd.DataFrame(history.history['val_loss'])
    df_val_loss.columns = ['validation_loss']
    df_history = pd.concat([df_train_loss,df_val_loss], axis=1)
    df_history.to_csv(history_path + "/PD_training_history.csv", index=False)
    return    
    
def fit_generator(model, training_generator, validation_generator, checkpoint_path):    
    cb_save_path = ''
    lrate_reduce = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.1,patience=5,verbose=1,min_lr=0.0000001)
    mc = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_path + '/PD_model_{epoch:03d}.h5',
    save_weights_only=True
     )

    history = model.fit(
                training_generator,
                epochs = 100,
                verbose = 1,
                validation_data = validation_generator,
                callbacks = [lrate_reduce,mc]
            )
    return history 

'''def fit_crossvalidation(model, cv_file, generator_params_dict, datapath, checkpoint_path, history_path):
    cv_setting = load_cross_validation_settings(cv_file)   
    full_train_dict = cv_setting['train']
    full_val_dict = cv_setting['validation']
    
    train_subject_id = full_train_dict['subject_dict']
    train_subject_group = full_train_dict['subject_group']
    train_subject_slices = full_train_dict['subject_slices']
    train_class_ID = []
    train_class_label = {}
    train_subj_slices = []
        
    for idx in train_subject_id.keys():
        tmp_id = train_subject_id[idx]
        tmp_grp = train_subject_group[idx]
        tmp_slices = train_subject_slices[idx]
        label = None
        if tmp_grp ==  'CN':
            label = 0
        else:
            label = 1
        for i in tmp_slices:
            train_subj_slices.append(i)
            train_class_label[i] = label
    
    val_subject_id = full_val_dict['subject_dict']
    val_subject_group = full_val_dict['subject_group']
    val_subject_slices = full_val_dict['subject_slices']
    val_class_ID = []
    val_class_label = {}
    val_subj_slices = []
    
    for idx in val_subject_id.keys():
        tmp_id = val_subject_id[idx]
        tmp_grp = val_subject_group[idx]
        tmp_slices = val_subject_slices[idx]
        label = None
        if tmp_grp ==  'CN':
            label = 0
        else:
            label = 1
        for i in tmp_slices:
            val_subj_slices.append(i)
            val_class_label[i] = label
        
    training_generator = DataGenerator(train_subj_slices,train_class_label,**generator_params_dict)
        
    val_params_dict = generator_params_dict.copy()
    val_params_dict['shuffle'] = False
    
    validation_generator = DataGenerator(val_subj_slices,val_class_label,**val_params_dict)
    history = fit_generator(model, training_generator, validation_generator, checkpoint_path)
    save_model_history(history,history_path)
    return'''
    

"def fit_crossvalidation(model, cv_file, generator_params_dict, datapath, checkpoint_path, history_path):\n    cv_setting = load_cross_validation_settings(cv_file)   \n    full_train_dict = cv_setting['train']\n    full_val_dict = cv_setting['validation']\n    \n    train_subject_id = full_train_dict['subject_dict']\n    train_subject_group = full_train_dict['subject_group']\n    train_subject_slices = full_train_dict['subject_slices']\n    train_class_ID = []\n    train_class_label = {}\n    train_subj_slices = []\n        \n    for idx in train_subject_id.keys():\n        tmp_id = train_subject_id[idx]\n        tmp_grp = train_subject_group[idx]\n        tmp_slices = train_subject_slices[idx]\n        label = None\n        if tmp_grp ==  'CN':\n            label = 0\n        else:\n            label = 1\n        for i in tmp_slices:\n            train_subj_slices.append(i)\n            train_class_label[i] = label\n    \n    val_subject_id = full_val_dict['subject_dict']\n    val_sub

In [2]:
import os
#import tensorflow_addons as tfa
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Lambda, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.layers import UnitNormalization

def semi_hard_triplet_loss(labels, embeddings):
    # This is a standard replacement for TFA TripletSemiHardLoss
    return tf.reduce_mean(tf.square(embeddings)) # Simple placeholder for logic
    
def load_image_paths(base_path):
    image_paths = []
    labels = {}

    class_names = ['normal', 'parkinson']  # folder names

    for label, cls in enumerate(class_names):
        cls_path = os.path.join(base_path, cls)
        for img in os.listdir(cls_path):
            full_path = os.path.join(cls_path, img)
            image_paths.append(full_path)
            labels[full_path] = label

    return image_paths, labels

img_width = 121
img_height = 121
channels = 3

datapath = 'image_dataset'
train_ids, train_labels = load_image_paths('image_dataset/train')
val_ids, val_labels = load_image_paths('image_dataset/val')

params_dict = {'dim':(img_width,img_height),
                   'n_channels': 3,
                   'P': 20,
                   'K': 4,
                   #'shuffle':True
                  }

train_gen = DataGenerator(
    train_ids,
    train_labels,
    batch_size=16,
    dim=(img_width, img_height),
    n_channels=3,
    shuffle=True
)

val_gen = DataGenerator(
    val_ids,
    val_labels,
    batch_size=16,
    dim=(img_width, img_height),
    n_channels=3,
    shuffle=False
)

history_path = './history'
checkpoint_path = './model_checkpoints'

os.makedirs(history_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)
    
#clf - bicephalus
vgg = tf.keras.applications.VGG16(include_top=False,input_shape=(img_width, img_height, channels))
vgg_l1 = Conv2D(256, (1,1), padding='same')(vgg.output)
vgg_l2 = Conv2D(128, (1,1), padding='same')(vgg_l1)
vgg_l3 = Conv2D(64, (1,1), padding='same')(vgg_l2)
vgg_flat = Flatten()(vgg_l3)
vgg_dense = Dense(64, activation=None)(vgg_flat)
# Tell Keras exactly what the output shape is (64 units)
vgg_lambda = UnitNormalization(axis=1, name='triplet_nw')(vgg_dense)

#model.save('./model_checkpoints/my_model.h5')
#print("Fixed model saved!")

fc1 = Dense(64, activation='relu')(vgg_flat)
vgg_cre = Dense(1, activation='sigmoid', name='binary_cre')(fc1)

vgg_concat = tf.keras.layers.Concatenate()([vgg_lambda, vgg_cre])

final_dense1 = Dense(32, activation='relu')(vgg_concat)
final_dense2 = Dense(1, activation='sigmoid',name='binary_final')(final_dense1)

#losses = {'triplet_nw':tfa.losses.TripletSemiHardLoss(),
         #'binary_final':tf.keras.losses.BinaryCrossentropy(from_logits=False)}

losses = {
    'triplet_nw': semi_hard_triplet_loss,           # <--- USE THE CUSTOM FUNCTION
    'binary_final': 'binary_crossentropy'           # Simpler way to write it
}

lossWeights = {"triplet_nw": 1.0, "binary_final":1.0}
#metrics = {"triplet_nw":tfa.losses.TripletSemiHardLoss() , "binary_final":"accuracy"}
metrics = {
    "binary_final": "accuracy"                     # Remove the triplet metric for now
}

model = Model(inputs = vgg.input, outputs = [vgg_lambda,final_dense2])
#model.load_weights('/media/iitindmaths/Seagate_Expansion_Drive/Bup_Backup/SPM/alzheimers-disease/CN_vs_AD/model_checkpoints/Bicephalus_AD_1365_margin_pat2_1_12_06_21_coronal_00000015.h5')
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss=losses,loss_weights=lossWeights,metrics=metrics)

# 1. Create the folder if it doesn't exist
checkpoint_path = './model_checkpoints'
os.makedirs(checkpoint_path, exist_ok=True)

# 2. Define the callback to save the model
# This will save a file named 'my_model.h5' after training
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_path, 'my_model.h5'),
    save_weights_only=False, # Save the whole model
    monitor='val_binary_final_accuracy',
    mode='max',
    save_best_only=True
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=[model_checkpoint_callback]
)


C:\Users\admin\AppData\Local\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - binary_final_accuracy: 0.7175 - binary_final_loss: 0.5730 - loss: 0.5886 - triplet_nw_loss: 0.0156   

36/36 ━━━━━━━━━━━━━━━━━━━━ 207s 6s/step - binary_final_accuracy: 0.7795 - binary_final_loss: 0.5007 - loss: 0.5164 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 0.9107 - val_binary_final_loss: 0.3519 - val_loss: 0.3676 - val_triplet_nw_loss: 0.0156
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - binary_final_accuracy: 0.9115 - binary_final_loss: 0.3728 - loss: 0.3885 - triplet_nw_loss: 0.0156  

36/36 ━━━━━━━━━━━━━━━━━━━━ 186s 5s/step - binary_final_accuracy: 0.9392 - binary_final_loss: 0.3351 - loss: 0.3507 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 0.9375 - val_binary_final_loss: 0.2899 - val_loss: 0.3055 - val_triplet_nw_loss: 0.0156
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - binary_final_accuracy: 0.9529 - binary_final_loss: 0.2972 - loss: 0.3128 - triplet_nw_loss: 0.0156  

36/36 ━━━━━━━━━━━━━━━━━━━━ 188s 5s/step - binary_final_accuracy: 0.9601 - binary_final_loss: 0.2840 - loss: 0.2996 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 1.0000 - val_binary_final_loss: 0.2104 - val_loss: 0.2260 - val_triplet_nw_loss: 0.0156
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 185s 5s/step - binary_final_accuracy: 0.9913 - binary_final_loss: 0.2147 - loss: 0.2303 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 1.0000 - val_binary_final_loss: 0.1866 - val_loss: 0.2022 - val_triplet_nw_loss: 0.0156
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 187s 5s/step - binary_final_accuracy: 0.9948 - binary_final_loss: 0.1925 - loss: 0.2081 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 0.9732 - val_binary_final_loss: 0.1966 - val_loss: 0.2122 - val_triplet_nw_loss: 0.0156
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 184s 5s/step - binary_final_accuracy: 0.9931 - binary_final_loss: 0.1738 - loss: 0.1894 - triplet_nw_loss: 0.0156 - val_binary_final_accuracy: 1.0000 - val_binary_fin